# Stage 14 data - new reranker training documents

Streams the NQ train split past the Stage 8 and Stage 11 windows, excludes their
titles and keeps the next 6000 documents.
Details: `docs/stage14_data_scale.md`.

## Setup

In [ ]:
# project folder on Drive; every account must see the shared folder at this path
PROJECT_DIR = '/content/drive/MyDrive/RAG chunk optimize'
ACCOUNT_LABEL = 'A'          # recorded in the run lock and the progress log
CLEAR_STALE_LOCK = False     # True only after confirming the runtime holding the lock is stopped

from google.colab import drive
drive.mount('/content/drive')

import os, shlex, subprocess, sys
if not os.path.isfile(os.path.join(PROJECT_DIR, 'config.py')):
    raise RuntimeError(f'no config.py under {PROJECT_DIR!r} - the shared folder is not mounted at this path.')
os.environ['RAG_DATA_ROOT'] = PROJECT_DIR + '/artifacts'
os.environ['PYTHONUNBUFFERED'] = '1'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
GUARD = f' --account {ACCOUNT_LABEL}' + (' --clear-stale-lock' if CLEAR_STALE_LOCK else '')


def run(cmd):
    """Stream the command's output live; a failure stops Run All."""
    proc = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'command failed: {cmd}')

## Install dependencies

In [ ]:
run('pip install -q -r requirements.txt')

## Preflight

Shared-folder id, write access, run lock and progress so far.

In [ ]:
run('python -u scripts/35_preflight_stage14.py')

## Stream the documents

Progress prints every 500 rows. The stream is not checkpointed, so an interrupted
run starts over.

In [ ]:
run('python -u scripts/32_build_stage14_data.py --stream-only' + GUARD)